# Day 10: Structured Output and Workflow Patterns

Welcome to the advanced half of the course.

Let me start with a question I ask every batch: **does every AI app need to be an agent?** Most people say yes. The honest answer is no, and most production apps aren't.

There are two ways to organise an LLM app:

- A **workflow**: *you* decide the steps. "First summarize, then translate, then check the length." The path is fixed in your graph.
- An **agent**: *the LLM* decides the steps, like the tool-calling loop from Day 6.

Workflows are cheaper, faster, easier to test, and much more predictable. Agents are flexible, but harder to control. A good rule: use a workflow whenever you can describe the steps in advance, and use an agent only when you truly can't.

Today we learn the workflow patterns that experienced teams use again and again. But first, one skill that makes all of them work properly: getting the LLM to return **clean, structured data** instead of free text.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. Structured output: a form instead of an essay

Suppose you ask an LLM to read a customer review and tell you whether it's positive. You might get "Positive", or "The review is mostly positive.", or "Overall, I'd say the sentiment leans positive, although...". Now try writing an `if` statement against that.

It's like asking customers for feedback on a blank sheet of paper versus giving them a form with checkboxes. The form is much easier to process.

`llm.with_structured_output(MyModel)` gives the LLM a form to fill in. You describe the form with a Pydantic class, and you get back a proper Python object with exactly those fields. The `Field(description=...)` text tells the LLM what each field means, so write it clearly.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
class Review(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="the overall feeling of the review")
    rating: int = Field(description="a star rating from 1 to 5")
    key_points: list[str] = Field(description="the main points, at most 3, each very short")

In [ ]:
review_llm = llm.with_structured_output(Review)

### Predict before you run

What type will `result` be? A string, a dictionary, or something else?

<details>
<summary>Click to see the answer</summary>

A `Review` object, an instance of the Pydantic class we wrote. So you can do `result.rating` and get an `int`, and `result.sentiment` is guaranteed to be one of the three allowed words.
</details>

In [ ]:
result = review_llm.invoke("Food at this Pune restaurant was amazing and cheap, but the service was really slow.")

print(type(result).__name__)
print("sentiment: ", result.sentiment)
print("rating:    ", result.rating)
print("key points:", result.key_points)

Now you can write normal code against it, like `if result.rating <= 2: alert_the_manager()`. And this is exactly how we'll let the LLM make **routing decisions** in a graph.

### Your turn

Add a field `would_return: bool` to `Review` with a description, and run it again on a review of your own.

In [ ]:
class ReviewV2(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="the overall feeling of the review")
    rating: int = Field(description="a star rating from 1 to 5")
    would_return: bool = Field(description="does the customer sound like they would come back?")

In [ ]:
my_review = "The biryani was cold and the waiter was rude. Never again."     # write your own
print(llm.with_structured_output(ReviewV2).invoke(my_review))

## 2. Pattern one: prompt chaining

Break a task into steps, where each LLM call works on the output of the previous one. It's an assembly line: one station does one job well.

The useful addition is a **gate**: a plain Python check between steps. If the output is already good, skip the next step. If not, fix it.

Example: write a product tagline. If it's longer than 6 words, send it to a `shorten` step.

```
  START -> write_tagline -> (more than 6 words?) --no--> END
                                    |
                                   yes
                                    v
                                 shorten -> END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class TaglineState(TypedDict):
    product: str
    tagline: str

In [ ]:
def write_tagline(state: TaglineState) -> dict:
    tagline = llm.invoke(f"Write a catchy tagline for: {state['product']}. Reply with the tagline only.").content
    return {"tagline": tagline}

In [ ]:
def too_long(state: TaglineState) -> Literal["shorten", "__end__"]:
    # The gate is plain Python. No LLM needed to count words.
    if len(state["tagline"].split()) > 6:
        return "shorten"
    return END

In [ ]:
def shorten(state: TaglineState) -> dict:
    shorter = llm.invoke(f"Shorten this to at most 6 words, keep it catchy: {state['tagline']}").content
    return {"tagline": shorter}

In [ ]:
builder = StateGraph(TaglineState)

In [ ]:
builder.add_node("write_tagline", write_tagline)
builder.add_node("shorten", shorten)

In [ ]:
builder.add_edge(START, "write_tagline")
builder.add_conditional_edges("write_tagline", too_long)
builder.add_edge("shorten", END)

In [ ]:
tagline_app = builder.compile()
show_graph(tagline_app)

In [ ]:
result = tagline_app.invoke({"product": "an app that reminds you to drink water"})
print(result["tagline"])

## 3. Pattern two: routing

On Day 3 we routed support tickets with keyword checks. That breaks easily: "I was charged twice" has no word "refund" in it. Now we let an LLM do the classifying, using structured output so the answer is always one of our team names.

Think of the receptionist at a hospital. They don't treat you; they listen for ten seconds and send you to the right department.

```
                 +--> billing --+
  START -> route +--> tech -----+--> END
                 +--> general --+
```

In [ ]:
class Route(BaseModel):
    team: Literal["billing", "tech", "general"] = Field(description="which team should handle this message")

In [ ]:
router_llm = llm.with_structured_output(Route)

In [ ]:
class SupportState(TypedDict):
    message: str
    team: str
    reply: str

In [ ]:
def route(state: SupportState) -> dict:
    decision = router_llm.invoke(f"Classify this customer message: {state['message']}")
    return {"team": decision.team}

In [ ]:
def billing(state: SupportState) -> dict:
    reply = llm.invoke(f"You are the billing team. Be precise about money. Reply in 2 sentences to: {state['message']}")
    return {"reply": reply.content}

In [ ]:
def tech(state: SupportState) -> dict:
    reply = llm.invoke(f"You are the tech support team. Give one clear troubleshooting step. Reply in 2 sentences to: {state['message']}")
    return {"reply": reply.content}

In [ ]:
def general(state: SupportState) -> dict:
    reply = llm.invoke(f"You are customer support. Be warm and friendly. Reply in 2 sentences to: {state['message']}")
    return {"reply": reply.content}

In [ ]:
def pick_team(state: SupportState) -> Literal["billing", "tech", "general"]:
    return state["team"]

In [ ]:
builder = StateGraph(SupportState)

In [ ]:
builder.add_node("route", route)
builder.add_node("billing", billing)
builder.add_node("tech", tech)
builder.add_node("general", general)

In [ ]:
builder.add_edge(START, "route")
builder.add_conditional_edges("route", pick_team)
builder.add_edge("billing", END)
builder.add_edge("tech", END)
builder.add_edge("general", END)

In [ ]:
support_app = builder.compile()
show_graph(support_app)

In [ ]:
messages = [
    "I was charged twice for my order",
    "The app freezes on the login screen",
    "Do you have a store in Indore?",
]

for message in messages:
    result = support_app.invoke({"message": message})
    print("Customer:", message)
    print("Team:    ", result["team"])
    print("Reply:   ", result["reply"])
    print()

## 4. Pattern three: parallelization

If several steps don't depend on each other, run them **at the same time**, then combine the results. Like a group project where three people each research one part, then one person writes the conclusion.

It's faster, and each LLM call stays focused on one small job, which usually gives better results than one giant prompt.

```
            +--> pros ---+
  START ----+--> cons ---+--> combine --> END
            +--> risks --+
```

In [ ]:
class DecisionState(TypedDict):
    question: str
    pros: str
    cons: str
    risks: str
    final: str

In [ ]:
def pros(state: DecisionState) -> dict:
    return {"pros": llm.invoke(f"List 3 short pros of: {state['question']}").content}

In [ ]:
def cons(state: DecisionState) -> dict:
    return {"cons": llm.invoke(f"List 3 short cons of: {state['question']}").content}

In [ ]:
def risks(state: DecisionState) -> dict:
    return {"risks": llm.invoke(f"List 2 short hidden risks of: {state['question']}").content}

In [ ]:
def combine(state: DecisionState) -> dict:
    prompt = (
        f"Question: {state['question']}\n"
        f"Pros: {state['pros']}\n"
        f"Cons: {state['cons']}\n"
        f"Risks: {state['risks']}\n"
        "Give a balanced recommendation in 2 sentences."
    )
    return {"final": llm.invoke(prompt).content}

In [ ]:
builder = StateGraph(DecisionState)

In [ ]:
builder.add_node("pros", pros)
builder.add_node("cons", cons)
builder.add_node("risks", risks)
builder.add_node("combine", combine)

In [ ]:
builder.add_edge(START, "pros")
builder.add_edge(START, "cons")
builder.add_edge(START, "risks")
builder.add_edge(["pros", "cons", "risks"], "combine")    # wait for all three, then combine
builder.add_edge("combine", END)

In [ ]:
decision_app = builder.compile()
show_graph(decision_app)

In [ ]:
result = decision_app.invoke({"question": "Should a fresher learn LangGraph before getting their first job?"})
print(result["final"])

Notice `add_edge(["pros", "cons", "risks"], "combine")`. A list on the left means "run `combine` only after **all** of these have finished".

Also notice we didn't need a reducer this time. Each parallel node writes a **different** key, so there's nothing to combine. Compare that with Day 2, where parallel nodes wrote the same key.

## 5. Pattern four: evaluator and optimizer

This is how a good writer works with an editor. The writer writes a draft. The editor reads it against a checklist and says "fix this". The writer revises. Repeat until the editor is happy, or until the deadline.

In our graph, one LLM call **writes**, another **grades** with structured output (pass or fail, plus feedback), and we loop. And because this is a loop, we add a maximum number of attempts, remember Day 3.

```
  START --> write --> evaluate --(pass, or 3 tries)--> END
              ^           |
              +---fail----+   (with feedback)
```

In [ ]:
class Grade(BaseModel):
    grade: Literal["pass", "fail"] = Field(description="pass only if every rule is followed")
    feedback: str = Field(description="what to fix if it failed, otherwise empty")

In [ ]:
grader = llm.with_structured_output(Grade)

RULES = "Exactly 4 lines. The lines must rhyme. It must mention the word 'chai'."

In [ ]:
class PoemState(TypedDict):
    topic: str
    poem: str
    grade: str
    feedback: str
    attempts: int

In [ ]:
def write(state: PoemState) -> dict:
    prompt = f"Write a poem about {state['topic']}. Rules: {RULES}"

    # On a second or third try, include the previous poem and what to fix
    if state.get("feedback"):
        prompt += f"\n\nYour previous poem:\n{state['poem']}\n\nFix this: {state['feedback']}"

    poem = llm.invoke(prompt).content
    return {"poem": poem, "attempts": state.get("attempts", 0) + 1}

In [ ]:
def evaluate(state: PoemState) -> dict:
    result = grader.invoke(f"Rules: {RULES}\nGrade this poem strictly:\n\n{state['poem']}")
    return {"grade": result.grade, "feedback": result.feedback}

In [ ]:
def good_enough(state: PoemState) -> Literal["write", "__end__"]:
    if state["grade"] == "pass":
        return END
    if state["attempts"] >= 3:
        return END              # stop trying, even if it isn't perfect
    return "write"

In [ ]:
builder = StateGraph(PoemState)

In [ ]:
builder.add_node("write", write)
builder.add_node("evaluate", evaluate)

In [ ]:
builder.add_edge(START, "write")
builder.add_edge("write", "evaluate")
builder.add_conditional_edges("evaluate", good_enough)

In [ ]:
poet = builder.compile()
show_graph(poet)

### Predict before you run

Each round makes two LLM calls: one to write, one to grade. In the worst case, where the grader never says "pass", how many LLM calls will this graph make in total?

<details>
<summary>Click to see the answer</summary>

Six. Three attempts, and each attempt is one `write` plus one `evaluate`. After the third grade, `good_enough` sees `attempts >= 3` and stops.

This is why the limit matters: without it, the worst case is "forever".
</details>

In [ ]:
result = poet.invoke({"topic": "rainy evenings"})

print(f"Attempts: {result['attempts']}, final grade: {result['grade']}\n")
print(result["poem"])

## 6. Common mistakes

**Mistake 1: Vague field descriptions.**
`rating: int` with no description might come back as 8 out of 10, or 80 out of 100. Say exactly what you want: "a star rating from 1 to 5". The descriptions are part of your prompt.

**Mistake 2: Using `str` when only a few answers are allowed.**
If a field can only be "billing", "tech" or "general", use `Literal[...]`. With plain `str`, sooner or later the model returns "Billing Team" and your router breaks.

**Mistake 3: Forgetting the attempt limit in an evaluator loop.**
A strict grader and a weak writer can disagree forever, and you pay for every round. Always cap the attempts.

**Mistake 4: Two parallel nodes writing the same key.**
In the parallel pattern, give each branch its own key (like `pros`, `cons`, `risks`). If they must share one, give that key a reducer (Day 2). Otherwise you'll get an `InvalidUpdateError`.

**Mistake 5: Reading a structured result like a dictionary.**
`result["rating"]` fails on a Pydantic object. Use `result.rating`.

In [ ]:
review = review_llm.invoke("Great food, decent price.")

try:
    print(review["rating"])
except TypeError as error:
    print("TypeError:", error)

print("The right way:", review.rating)

## 7. Practice

**Exercise 1.** Pull a structured `Invoice` out of this text: *"Hi, please pay Rs. 12,500 to Sharma Electricals by 15 Nov 2026."* Fields: `vendor` (str), `amount` (float), `currency` (str, a 3-letter code), `due_date` (str in YYYY-MM-DD).

**Exercise 2.** Build a parallel graph that translates one sentence into Hindi, Tamil and Bengali at the same time, then a `combine` node puts all three together.

In [ ]:
# Solution 1
class Invoice(BaseModel):
    vendor: str
    amount: float
    currency: str = Field(description="3-letter currency code, for example INR")
    due_date: str = Field(description="date in YYYY-MM-DD format")

In [ ]:
invoice = llm.with_structured_output(Invoice).invoke(
    "Hi, please pay Rs. 12,500 to Sharma Electricals by 15 Nov 2026."
)
print(invoice)

In [ ]:
# Solution 2
class TranslationState(TypedDict):
    text: str
    hindi: str
    tamil: str
    bengali: str
    combined: str

In [ ]:
def to_hindi(state: TranslationState) -> dict:
    return {"hindi": llm.invoke(f"Translate into Hindi. Reply with only the translation: {state['text']}").content}

In [ ]:
def to_tamil(state: TranslationState) -> dict:
    return {"tamil": llm.invoke(f"Translate into Tamil. Reply with only the translation: {state['text']}").content}

In [ ]:
def to_bengali(state: TranslationState) -> dict:
    return {"bengali": llm.invoke(f"Translate into Bengali. Reply with only the translation: {state['text']}").content}

In [ ]:
def combine_translations(state: TranslationState) -> dict:
    combined = f"Hindi:   {state['hindi']}\nTamil:   {state['tamil']}\nBengali: {state['bengali']}"
    return {"combined": combined}

In [ ]:
builder = StateGraph(TranslationState)

In [ ]:
builder.add_node("to_hindi", to_hindi)
builder.add_node("to_tamil", to_tamil)
builder.add_node("to_bengali", to_bengali)
builder.add_node("combine", combine_translations)

In [ ]:
builder.add_edge(START, "to_hindi")
builder.add_edge(START, "to_tamil")
builder.add_edge(START, "to_bengali")
builder.add_edge(["to_hindi", "to_tamil", "to_bengali"], "combine")
builder.add_edge("combine", END)

In [ ]:
translation_app = builder.compile()

In [ ]:
result = translation_app.invoke({"text": "Practice a little every day."})
print(result["combined"])

## Wrapping up

`with_structured_output` turns the LLM's answer into a typed Python object, and that's what makes LLM-driven decisions reliable. On top of that we built four patterns you'll reuse constantly:

- **Chaining**: steps in a row, with plain-Python gates between them
- **Routing**: an LLM classifier picks the branch
- **Parallelization**: independent steps at once, then `add_edge([...], "combine")`
- **Evaluator and optimizer**: write, grade, improve, with an attempt limit

All of these had a **fixed** number of branches. But what if you don't know in advance how many parallel tasks you'll need, like "summarize each of the files the user uploaded"? Tomorrow: `Send` and the map-reduce pattern.